# CardioSense Phase 1 — Pipeline 2: ECG Interpretation

**PTB-XL → signal preprocessing → statistical baseline → 1D CNN → Integrated Gradients**

Run `00_colab_setup.ipynb` first, including section 7b (the PTB-XL download).

**This notebook needs a GPU.** `Runtime → Change runtime type → GPU`.

### Sections
1. Environment Setup · 2. Imports · 3. Configuration · 4. Dataset Verification
5. Exploratory Data Analysis · 6. Preprocessing · 7. Dataset Splitting · 8. Baseline
9. Model Definition · 10. Training · 11. Validation · 12. Evaluation
13. Explainability · 14. *(calibration — see note)* · 15. Error Analysis
16. Save Model · 17. Save Results · 18. Example Inference

### The task, stated precisely

**5-class diagnostic superclass classification — MULTI-LABEL.**

`NORM`, `MI`, `STTC`, `CD`, `HYP`. A single ECG can legitimately carry several of
these at once, so this is five independent binary decisions, not a choice among
five options. Everything downstream follows from that:

| Consequence | Why |
|---|---|
| `BCEWithLogitsLoss`, not cross-entropy | Softmax would force the five scores to compete for one probability budget |
| One threshold **per class** | Prevalence runs from ~12% (HYP) to ~44% (NORM) |
| **Accuracy is never reported** | Undefined for multi-label; exact-match ratio is reported under that name |
| Macro ROC-AUC is the headline | Weights all five classes equally, so rare classes cannot be hidden |

### Experiments
| ID | Experiment | Question |
|---|---|---|
| E-A | Statistical features + one-vs-rest LogReg | What floor must deep learning clear? |
| E-B | 1D CNN | Does temporal structure help beyond global statistics? |
| E-C | ResNet-1D *(optional)* | Does extra depth earn its cost? |

**On disconnects:** every epoch writes `last.pt`. If Colab drops, just re-run the
training cell — it resumes from the next epoch, it does not restart.

## 1. Environment Setup

In [ ]:
import os, sys, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)

    REPO_DIR = Path("/content/CardioSense")
    if not REPO_DIR.exists():
        raise RuntimeError("Repo not found. Run 00_colab_setup.ipynb first.")
    os.chdir(REPO_DIR)

    os.environ["CARDIOSENSE_DATA_ROOT"] = "/content/drive/MyDrive/CardioSense/data"
    subprocess.run(["pip", "install", "-q", "-r", "requirements-colab.txt"], check=False)
    subprocess.run(["pip", "install", "-q", "-e", "."], check=False)
else:
    here = Path.cwd()
    while not (here / "pyproject.toml").exists() and here != here.parent:
        here = here.parent
    os.chdir(here)

print("Working directory:", Path.cwd())
print("Data root        :", os.environ.get("CARDIOSENSE_DATA_ROOT", "<repo>/data"))

## 2. Imports

In [ ]:
import json
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from IPython.display import Image, display

from cardiosense.common.config import load_config
from cardiosense.common.env import get_device, print_environment
from cardiosense.common.experiment import ExperimentTracker, load_experiment_log
from cardiosense.common.io_utils import save_json, save_pickle
from cardiosense.common.paths import PATHS
from cardiosense.common.plots import plot_class_distribution, plot_training_curves
from cardiosense.common.seeding import set_seed
from cardiosense.common.training import count_parameters

from cardiosense.ecg.data import (
    build_superclass_labels, describe_dataset, load_metadata,
    parse_scp_codes, resolve_ptbxl_root, split_by_fold, verify_dataset,
)
from cardiosense.ecg.preprocessing import (
    build_waveform_cache, load_raw_record, preprocess_signal, remove_baseline_wander,
)
from cardiosense.ecg.dataset import build_dataloaders, compute_pos_weight
from cardiosense.ecg.baseline import extract_feature_matrix, feature_names, train_baseline
from cardiosense.ecg.models import build_model, model_summary
from cardiosense.ecg.trainer import train_model
from cardiosense.ecg import evaluate as ev
from cardiosense.ecg.explain import run_ig_analysis
from cardiosense.ecg.predict import ECGPredictor

pd.set_option("display.width", 150)
pd.set_option("display.max_columns", 40)
_ = print_environment()

## 3. Configuration

Everything comes from `configs/ecg_config.yaml`. Note `sampling_rate: 100` — PTB-XL
ships every record at both 100 Hz and 500 Hz, and we read the native 100 Hz files
rather than downsampling, so no resampling artefacts are introduced.

In [ ]:
cfg = load_config("ecg")

SEED = int(cfg.seed)
set_seed(SEED, strict=bool(cfg.get("strict_determinism", False)))
device = get_device()

RESULTS = PATHS.root / cfg.output.results_dir
MODELS = PATHS.root / cfg.output.models_dir
CHECKPOINTS = MODELS / "checkpoints"
for path in (RESULTS, MODELS, CHECKPOINTS):
    path.mkdir(parents=True, exist_ok=True)

CLASSES = list(cfg.task.classes)

print(f"task          : {cfg.task.name} ({cfg.task.type})")
print(f"classes       : {CLASSES}")
print(f"sampling rate : {cfg.dataset.sampling_rate} Hz, {cfg.dataset.signal_length} samples "
      f"({cfg.dataset.signal_length / cfg.dataset.sampling_rate:.0f} s), "
      f"{cfg.dataset.n_leads} leads")
print(f"split         : folds {cfg.split.train_folds} / {cfg.split.val_folds} / "
      f"{cfg.split.test_folds}")
print(f"model         : {cfg.model.name} | batch {cfg.training.batch_size} | "
      f"lr {cfg.training.learning_rate} | epochs {cfg.training.epochs}")
print(f"device        : {device}")

## 4. Dataset Verification

PTB-XL is open access — no credentialing form. If this cell fails, run section 7b
of `00_colab_setup.ipynb`.

In [ ]:
root = resolve_ptbxl_root(cfg)
print(json.dumps(verify_dataset(cfg, root), indent=2))

database, statements = load_metadata(cfg, root)
print(f"\nrecords : {len(database)}")
print(f"patients: {database.patient_id.nunique()}")
print(f"folds   : {sorted(database.strat_fold.unique())}")
database[["patient_id", "age", "sex", "scp_codes", "strat_fold", "filename_lr"]].head()

In [ ]:
# scp_codes is a STRINGIFIED dict of {SCP code: likelihood 0-100}.
# A likelihood of 0.0 means "likelihood not stated" — it counts as PRESENT.
# Reading it as absent would silently delete a large share of the positive labels.
example = database.iloc[0]
print("raw string:", example.scp_codes)
print("parsed    :", parse_scp_codes(example.scp_codes))

diagnostic = statements[statements.diagnostic == 1]
print(f"\n{len(statements)} SCP statements, {len(diagnostic)} of them diagnostic")
print("superclasses:", sorted(diagnostic.diagnostic_class.dropna().unique()))
diagnostic[["diagnostic", "diagnostic_class", "diagnostic_subclass"]].head(8)

## 5. Label construction & Exploratory Data Analysis

In [ ]:
database, labels, label_report = build_superclass_labels(database, statements, cfg)
save_json(label_report, RESULTS / "label_report.json")

print(f"records kept: {label_report['n_records_out']} of {label_report['n_records_in']} "
      f"({label_report['records_without_any_superclass']} had no diagnostic superclass)")
print(f"patients    : {label_report['n_patients']}")
print(f"\nlabels per record: {label_report['labels_per_record']}")

pd.DataFrame({
    "support": label_report["support"],
    "prevalence": label_report["prevalence"],
}).T

In [ ]:
description = describe_dataset(database, labels, cfg)
save_json(description, RESULTS / "dataset_description.json")

plot_class_distribution(description["class_support"], RESULTS / "class_distribution.png",
                        title="Diagnostic superclass support (multi-label — bars overlap)")
display(Image(filename=str(RESULTS / "class_distribution.png")))

print(f"{100 * description['multi_label_fraction']:.1f}% of records carry MORE THAN ONE "
      "superclass.")
print("That is exactly why this is multi-label and why plain accuracy is not reported.\n")

# Co-occurrence: the diagonal is each class's support; off-diagonal entries are
# genuine simultaneous diagnoses.
co = pd.DataFrame(description["co_occurrence"])[CLASSES].loc[CLASSES]
fig, ax = plt.subplots(figsize=(6, 5))
image = ax.imshow(co.to_numpy(), cmap="Blues")
ax.set_xticks(range(len(CLASSES)), CLASSES)
ax.set_yticks(range(len(CLASSES)), CLASSES)
for i in range(len(CLASSES)):
    for j in range(len(CLASSES)):
        value = co.iloc[i, j]
        ax.text(j, i, f"{value:,}", ha="center", va="center", fontsize=9,
                color="white" if value > co.to_numpy().max() / 2 else "black")
ax.set_title("Superclass co-occurrence\n(off-diagonal = records with both labels)")
ax.grid(False)
fig.colorbar(image, ax=ax, shrink=0.8)
plt.show()

## 6. Signal Preprocessing

Each step is justified, because "standard ECG preprocessing" is not a
justification and a filter applied for no reason destroys real morphology.

| Step | On? | Why |
|---|---|---|
| 0.5 Hz high-pass | **yes** | Removes respiration/electrode drift, which is not diagnostic and wrecks per-lead normalisation. Zero-phase (`filtfilt`) so the ST segment is not shifted in time |
| Low-pass | **no** | The 100 Hz files are already anti-alias filtered; another low-pass would blunt QRS upstrokes, which is what `CD` is about |
| 50 Hz notch | **no** | Handled upstream, and meaningless at/beyond Nyquist for a 100 Hz signal |
| Per-lead z-score | **yes** | Removes electrode/body-habitus amplitude variation. Computed **within each record**, so it is leakage-free by construction |
| Resampling | **no** | We read native 100 Hz files |
| 8σ clipping | **yes** | Caps electrode pops without touching real QRS peaks (~3–5σ) |

**Known cost:** per-lead z-scoring discards absolute voltage, which matters for
`HYP` (partly an amplitude criterion). Recorded as a limitation, not glossed over.

In [ ]:
# Look at one record before and after preprocessing.
example_row = database.iloc[0]
raw_signal, header = load_raw_record(root / example_row.filename_lr)
processed = preprocess_signal(raw_signal, cfg)

print(f"raw       : shape {raw_signal.shape} (samples, leads), fs = {header['fs']} Hz")
print(f"processed : shape {processed.shape} (leads, samples)")
print(f"leads     : {header['sig_name']}")

fs = int(cfg.dataset.sampling_rate)
t = np.arange(processed.shape[-1]) / fs
lead = 1  # lead II

fig, axes = plt.subplots(2, 1, figsize=(13, 5), sharex=True)
axes[0].plot(t, raw_signal[:, lead], lw=0.8, color="tab:gray")
axes[0].set_title("Lead II — raw (note the slow baseline drift)")
axes[0].set_ylabel("mV")
axes[1].plot(t, processed[lead], lw=0.8, color="tab:blue")
axes[1].set_title("Lead II — after 0.5 Hz high-pass + per-lead z-score")
axes[1].set_ylabel("z")
axes[1].set_xlabel("time (s)")
for ax in axes:
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

### Build the waveform cache

Reading ~21,800 WFDB files is I/O-bound and slow, especially from Drive. We do it
once, preprocess, and save one contiguous `.npy` (~1.0 GB at 100 Hz) that is then
memory-mapped. Every later epoch reads from that instead of parsing files.

Baking preprocessing into the cache is safe **only** because every step is
per-record — no statistic is shared across records, so there is nothing to leak
between splits.

First run: several minutes. Later runs: instant.

In [ ]:
start = time.time()
waveforms, cache_path = build_waveform_cache(database, cfg, root=root)
print(f"waveforms: {waveforms.shape} ({waveforms.dtype}) in {time.time() - start:.1f}s")
print(f"cache    : {cache_path}")
print(f"size     : {cache_path.stat().st_size / 1024**3:.2f} GB")

## 7. Dataset Splitting — official `strat_fold`

PTB-XL ships a `strat_fold` column (1–10) assigned by the dataset authors:
stratified by diagnostic class **and patient-disjoint**.

Folds 1–8 train, 9 validation, 10 test. This is the published convention, so our
macro-AUC is comparable to the literature, and leakage is impossible by
construction.

**Never re-split PTB-XL randomly by record.** Patients contribute several ECGs; a
record-level shuffle puts the same patient on both sides of the split. The cell
below asserts patient disjointness rather than trusting it.

In [ ]:
splits = split_by_fold(database, labels, cfg)

split_summary = {name: splits[name]["summary"] for name in ("train", "val", "test")}
split_summary["patient_overlap"] = splits["patient_overlap"]
save_json(split_summary, RESULTS / "split_summary.json")

print("patient overlap between splits (must be all zero):", splits["patient_overlap"])
pd.DataFrame({
    name: {
        "records": splits[name]["summary"]["n_records"],
        "patients": splits[name]["summary"]["n_patients"],
        **{f"prev_{c}": splits[name]["summary"]["prevalence"][c] for c in CLASSES},
    } for name in ("train", "val", "test")
})

## 8. Baseline — Experiment E-A

11 statistics per lead (132 features), one-vs-rest Logistic Regression.

Its job is not to be good; it is to set the floor the CNN must clear. A network
that beats chance but not this has learned nothing a summary statistic could not.

**Structural prediction to check later:** these features are *global* over 10
seconds and discard temporal ordering entirely. ST elevation/depression is defined
by *where* a deflection sits relative to the QRS, so `STTC` and `MI` are where the
CNN should gain most. Worth verifying against the per-class numbers rather than
assuming.

In [ ]:
features = list(cfg.baseline.features)
print(f"{len(features)} features x {cfg.dataset.n_leads} leads = "
      f"{len(features) * cfg.dataset.n_leads} total")
print(features)

start = time.time()
X_train = extract_feature_matrix(waveforms, splits["train"]["indices"], features)
X_val = extract_feature_matrix(waveforms, splits["val"]["indices"], features)
X_test = extract_feature_matrix(waveforms, splits["test"]["indices"], features)
print(f"\nfeature matrices: {X_train.shape} / {X_val.shape} / {X_test.shape} "
      f"in {time.time() - start:.1f}s")

y_train = splits["train"]["labels"]
y_val = splits["val"]["labels"]
y_test = splits["test"]["labels"]

baseline = train_baseline(X_train, y_train, X_val, cfg, X_test=X_test)
thr_baseline, _ = ev.tune_per_class_thresholds(y_val, baseline["val_prob"], cfg)
metrics_baseline = ev.evaluate_multilabel(
    y_test, baseline["test_prob"], thr_baseline, cfg, split_name="test/baseline"
)

experiments = {"statistical_baseline": {
    "metrics": metrics_baseline,
    "parameters": int(X_train.shape[1] * len(CLASSES)),
    "train_seconds": baseline["train_seconds"],
}}
save_pickle(baseline["model"], MODELS / "ecg_baseline.pkl")
ev.per_class_table(metrics_baseline, CLASSES)

## 9. Model Definition — 1D CNN (Experiment E-B)

Four convolutional stages. Kernel size 7 at 100 Hz spans 70 ms — about the width
of a QRS complex, so a first-layer filter can respond to a whole complex rather
than a fragment. Each stage halves the time axis, so by the last stage the
receptive field covers several beats.

A 1D convolution slides along time across **all 12 leads at once**, which matches
the physiology: the leads are simultaneous views of the same electrical event, not
independent channels to be merged at the end.

Not deeper, because ~17k records with no pretraining will overfit a large network
before it generalises. The head emits **raw logits** — `BCEWithLogitsLoss` fuses
the sigmoid into the loss for numerical stability.

In [ ]:
loaders = build_dataloaders(waveforms, labels, splits, cfg)

pos_weight = None
if str(cfg.training.get("pos_weight", "auto")) == "auto":
    pos_weight = compute_pos_weight(y_train)
    print("pos_weight (n_neg / n_pos per class):",
          {c: round(float(w), 2) for c, w in zip(CLASSES, pos_weight)})
    print("Without this, the model can minimise loss by rarely predicting the rare classes.\n")

model = build_model(cfg).to(device)
architecture = model_summary(model, input_shape=(2, cfg.dataset.n_leads, cfg.dataset.signal_length))
print(json.dumps(architecture, indent=2))
print(model)

## 10. Training

Monitored metric is **`val_macro_auc`**, not validation loss. Loss is dominated by
the common classes, so it can improve while `HYP` — the rare and clinically
interesting one — degrades. Macro AUC weights all five classes equally and is
threshold-free.

**If Colab disconnects, just re-run this cell.** Every epoch writes `last.pt` with
model, optimiser, scheduler, AMP scaler and RNG state; training resumes at the
next epoch.

In [ ]:
training_result = train_model(
    model, loaders, cfg, device, CHECKPOINTS,
    pos_weight=pos_weight, experiment_name=str(cfg.model.name),
)

print(f"\nbest {training_result['monitor']} = {training_result['best_value']:.4f} "
      f"at epoch {training_result['best_epoch']}")
print(f"epochs run: {training_result['epochs_run']} "
      f"(resumed from epoch {training_result['resumed_from']})")
print(f"time      : {training_result['total_seconds']:.0f}s "
      f"({training_result['seconds_per_epoch']:.1f}s/epoch)")

In [ ]:
plot_training_curves(
    training_result["history"], RESULTS / "training_curve.png",
    loss_keys=("train_loss", "val_loss"),
    metric_keys=("val_macro_auc", "val_macro_pr_auc", "val_macro_f1"),
    best_epoch=training_result["best_epoch"],
    title=f"{cfg.model.name} — training history",
)
display(Image(filename=str(RESULTS / "training_curve.png")))

# Read the curves: train loss falling while val loss rises = overfitting.
# Both flat and high = underfitting (raise capacity or LR, or train longer).
pd.DataFrame(training_result["history"]).tail(10).round(4)

## 11. Validation — per-class thresholds

One threshold per class, tuned on **validation only**. A single global 0.5 is
wrong here: prevalence runs from ~12% to ~44%, and after `pos_weight` training the
logit scales differ between classes.

In [ ]:
use_amp = bool(cfg.training.get("amp", True)) and device.type == "cuda"

val_prob, val_true = ev.predict_probabilities(model, loaders["val"], device, use_amp)
thresholds, threshold_info = ev.tune_per_class_thresholds(val_true, val_prob, cfg)
save_json(threshold_info, RESULTS / "thresholds.json")

pd.DataFrame(threshold_info["per_class"]).T

## 12. Evaluation — test split, scored once

Metrics chosen for a multi-label task:

- **Macro ROC-AUC** (headline) — all five classes weighted equally
- **Macro PR-AUC** — chance level is each class's prevalence, marked on the plot
- **Macro / micro / weighted F1**
- **Exact-match ratio** — the multi-label analogue of accuracy, reported under
  that explicit name so nobody reads it as ordinary accuracy
- **Hamming loss**

Not reported: anything labelled "accuracy".

In [ ]:
test_prob, test_true = ev.predict_probabilities(model, loaders["test"], device, use_amp)
test_metrics = ev.evaluate_multilabel(test_true, test_prob, thresholds, cfg, split_name="test")

experiments[str(cfg.model.name)] = {
    "metrics": test_metrics,
    "parameters": count_parameters(model),
    "train_seconds": training_result["total_seconds"],
}

per_class = ev.per_class_table(test_metrics, CLASSES)
per_class.to_csv(RESULTS / "per_class_metrics.csv", index=False)
per_class

In [ ]:
ev.plot_per_class_curves(test_true, test_prob, CLASSES, RESULTS, prefix="test")
ev.plot_per_class_confusion(test_true, test_prob, thresholds, CLASSES, RESULTS)

for name in ["roc_curve.png", "pr_curve.png", "confusion_matrix.png"]:
    display(Image(filename=str(RESULTS / name)))

In [ ]:
# Does the deep model actually earn its complexity? The honest answer is
# sometimes no, and the code says so rather than burying it.
comparison = ev.build_comparison_table(experiments)
comparison.to_csv(RESULTS / "model_comparison.csv", index=False)
display(comparison)

recommendation = ev.recommend_model(experiments)
save_json(recommendation, RESULTS / "model_recommendation.json")
print(f"\nRECOMMENDED: {recommendation['recommended']}")
for reason in recommendation["reasoning"]:
    print("  -", reason)
if "warning" in recommendation:
    print("\nWARNING:", recommendation["warning"])

In [ ]:
# Where did the CNN gain over the baseline, class by class?
# Prediction from section 8: most of the gain should land on STTC and MI,
# because those depend on temporal position, which global statistics discard.
if "statistical_baseline" in experiments:
    gains = pd.DataFrame({
        "baseline_auc": [metrics_baseline["per_class"][c].get("roc_auc") for c in CLASSES],
        "cnn_auc": [test_metrics["per_class"][c].get("roc_auc") for c in CLASSES],
    }, index=CLASSES)
    gains["gain"] = (gains.cnn_auc - gains.baseline_auc).round(4)
    display(gains.round(4).sort_values("gain", ascending=False))

## 14. Calibration — deliberately not done here

The clinical pipeline fits a calibrator; this one does not, and that is a
considered decision rather than an omission.

Multi-label calibration means five independent calibrators, each fitted on the
validation split, each needing enough positives to be stable. It is doable, but it
is a Phase 2 concern: the fusion stage is where calibrated ECG confidences will
actually be *used*, and doing it properly deserves its own evaluation rather than
being tacked on here.

So the ECG probabilities are **sigmoid outputs — ranking scores, not frequency
claims**, and `predict.py` says so in its output. Phase 2 must calibrate them
before using them as fusion weights.

## 15. Error Analysis

In [ ]:
errors = ev.export_errors(
    splits["test"]["database"], test_true, test_prob, thresholds, CLASSES,
    RESULTS / "errors", prefix="test",
)

print(f"{errors['n_records_with_any_error']} of {errors['n_records']} records have "
      f"at least one class wrong (exact-match {errors['exact_match_rate']:.3f})")
pd.DataFrame(errors["per_class"]).T[["n_false_negative", "n_false_positive", "threshold"]]

In [ ]:
# Which class pairs get confused? For a multi-label task the interesting question
# is: when we falsely predict class A, which class was actually present?
predictions = (test_prob >= thresholds[None, :]).astype(int)
confusion = np.zeros((len(CLASSES), len(CLASSES)), dtype=int)
for predicted in range(len(CLASSES)):
    false_positive_rows = (predictions[:, predicted] == 1) & (test_true[:, predicted] == 0)
    for actual in range(len(CLASSES)):
        confusion[predicted, actual] = int(test_true[false_positive_rows, actual].sum())

fig, ax = plt.subplots(figsize=(6.5, 5.5))
ax.imshow(confusion, cmap="Oranges")
ax.set_xticks(range(len(CLASSES)), CLASSES)
ax.set_yticks(range(len(CLASSES)), CLASSES)
ax.set_xlabel("was actually present")
ax.set_ylabel("falsely predicted")
for i in range(len(CLASSES)):
    for j in range(len(CLASSES)):
        ax.text(j, i, confusion[i, j], ha="center", va="center", fontsize=9)
ax.set_title("False positives: what was really there?")
ax.grid(False)
plt.tight_layout()
plt.show()

## 13. Explainability — Integrated Gradients

**This is Integrated Gradients, and it is called that.** Neither architecture
contains an attention mechanism, so calling this "attention" would misdescribe the
model.

IG integrates the gradient along a straight path from a baseline (a flat trace) to
the actual signal. Its defining property is **completeness**: the attributions must
sum to `F(x) − F(baseline)`. The code checks this and automatically raises
`n_steps` when it fails, rather than returning a map that is known not to sum
correctly.

**Limitations, for the report:**
1. Attribution shows where the model was *sensitive*, not that a finding is present.
2. A different baseline redistributes the attributions; there is no uniquely correct choice.
3. Attributions are per-sample; the meaningful physiological unit is the beat.
4. Leads viewing overlapping territory (II/III/aVF) share credit arbitrarily.

In [ ]:
ig_summary = run_ig_analysis(
    model, waveforms, splits["test"]["indices"], test_true, test_prob,
    thresholds, cfg, device, RESULTS / "explanations",
)

print(f"{ig_summary['n_explanations']} explanations written")
pd.DataFrame([
    {"record": r["record_id"], "class": r["class"], "case": r["case_type"],
     "p": r["probability"], "true": r["true_label"],
     "top_lead": r["top_leads"][0]["lead"],
     "peak_window_s": r["peak_window_seconds"]}
    for r in ig_summary["explanations"]
])

In [ ]:
# Show a true positive and a false negative side by side. The false negative is
# the informative one: where did the model look when it MISSED the finding?
for case_type in ("TP", "FN"):
    for record in ig_summary["explanations"]:
        if record["case_type"] == case_type:
            print(f"--- {case_type}: class {record['class']}, p = {record['probability']}, "
                  f"true = {record['true_label']} ---")
            display(Image(filename=str(RESULTS / "explanations" / record["figure"])))
            break

## 16. Save Model

In [ ]:
model_path = MODELS / cfg.output.model_file
torch.save({
    "model_state": model.state_dict(),
    "model_name": str(cfg.model.name),
    "class_names": CLASSES,
    "thresholds": thresholds.tolist(),
}, model_path)

inference_config = {
    "model_version": str(cfg.output.model_version),
    "model_name": str(cfg.model.name),
    "model_params": cfg.model.get(str(cfg.model.name), {}).to_dict(),
    "classes": CLASSES,
    "label_mapping": {str(i): name for i, name in enumerate(CLASSES)},
    "thresholds": {name: round(float(t), 4) for name, t in zip(CLASSES, thresholds)},
    "task_type": str(cfg.task.type),
    "sampling_rate": int(cfg.dataset.sampling_rate),
    "signal_length": int(cfg.dataset.signal_length),
    "n_leads": int(cfg.dataset.n_leads),
    "lead_names": list(cfg.dataset.lead_names),
    "preprocessing": cfg.preprocessing.to_dict(),
}
config_path = save_json(inference_config, MODELS / cfg.output.config_file)

metadata_path = save_json({
    **inference_config,
    "created_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    "modality": "ecg",
    "dataset": {"name": str(cfg.dataset.name), "version": str(cfg.dataset.version)},
    "architecture": architecture,
    "split_summary": split_summary,
    "label_report": label_report,
    "training": training_result,
    "test_metrics": test_metrics,
    "threshold_info": threshold_info,
    "recommendation": recommendation,
    "training_config": cfg.to_dict(),
}, MODELS / cfg.output.metadata_file)

for path in (model_path, config_path, metadata_path):
    print(f"  {path.name:<28} {path.stat().st_size / 1024:>9.1f} KB")

## 17. Save Results

In [ ]:
save_json({
    "model": str(cfg.model.name),
    "test": test_metrics,
    "model_comparison": comparison.to_dict(orient="records"),
    "recommendation": recommendation,
    "split": split_summary,
    "thresholds": threshold_info,
    "training": training_result,
}, RESULTS / "metrics.json")

with ExperimentTracker("ecg_notebook", modality="ecg", config=cfg,
                       primary_metric="macro_roc_auc") as run:
    run.log_params({
        "model": str(cfg.model.name),
        "batch_size": int(cfg.training.batch_size),
        "learning_rate": float(cfg.training.learning_rate),
        "epochs": int(cfg.training.epochs),
        "parameters": architecture["total"],
        "n_train": split_summary["train"]["n_records"],
    })
    run.log_metrics({k: v for k, v in test_metrics.items() if isinstance(v, (int, float))},
                    split="test")
    run.log_history(training_result["history"])
    run.set_best_epoch(int(training_result["best_epoch"]))
    run.log_artifact("model", model_path)

print("\nfiles written:")
for path in sorted(RESULTS.rglob("*")):
    if path.is_file():
        print("  ", path.relative_to(RESULTS))

## 18. Example Inference

Loads the saved artifacts from scratch and runs on a raw WFDB record, the way a
Phase 2 caller would.

Note the output shape: **several classes can be positive at once**. It is
deliberately not collapsed into a single diagnosis.

In [ ]:
predictor = ECGPredictor.load()

test_record = splits["test"]["database"].iloc[0]
record_path = root / test_record.filename_lr

result = predictor.predict_record(record_path)
print(json.dumps({k: v for k, v in result.items() if k != "notes"}, indent=2))

actual = {c: int(splits["test"]["labels"][0, i]) for i, c in enumerate(CLASSES)}
print("\nactual labels:", actual)
print("predicted    :", result["predictions"])

## Phase 1 ECG checklist

| Item | Where |
|---|---|
| PTB-XL verified | §4 |
| Classification task selected & justified | §3, top of notebook, `docs/datasets.md` |
| Signal preprocessing | §6 — each step justified individually |
| Leakage-free split | §7 — official `strat_fold`, patient disjointness asserted |
| Baseline | §8 (E-A) |
| 1D CNN | §9–10 (E-B) |
| ResNet-1D if justified | `--set model.name=resnet1d` (E-C) |
| Evaluation | §12 — multi-label metrics, no "accuracy" |
| Error analysis | §15 |
| Explainability | §13 — Integrated Gradients, completeness-checked |
| Saved model | §16 |
| Inference script | §18 · `src/cardiosense/ecg/predict.py` |

### Running the ResNet-1D comparison (E-C)

Only worth doing once the CNN trains stably:

```bash
python -m cardiosense.ecg.train --set model.name=resnet1d --experiment-name ecg_resnet
```

Then compare against the CNN. `ev.recommend_model` applies a 0.01 macro-AUC bar —
below that, run-to-run seed variation explains the difference, and the honest
recommendation is to keep the simpler CNN.

**Next:** `03_xray_training.ipynb`.

Headless equivalent of this notebook:

```bash
python -m cardiosense.ecg.train
```

*Research artifact. Not a medical device. Not for clinical use.*